<a href="https://colab.research.google.com/github/peicheng2005/CBio/blob/Colab/PredictCE_from_chemicalStructure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import classification_report

In [ ]:

# CE stands for cytosolic exposure, an important property for a peptide compound to work inside a cell
# The code reads in  experimentally measured CE, the sequence and physiochemical properties, and exfp fingerprint,
# build a model to predict CE using XGBoost
# It evaluates the model, saves it as a pickle file to be deployed elsewhere,
# and ranks the important features

# read in the file with's already QC filtered, control removed, bad target removed,  replicate collapsed
halofile = '~/NewModels/compound_Sequence_and_CE.csv'
vectFile = "~/NewModels/compound_ChemicalStructure_Vectors_from_RDKit.csv"

df = pd.read_csv(halofile, header = 0, delimiter=',', skipinitialspace = True)
df3 =df[['Linear_Sequence','CE']]

df4 = pd.read_csv(vectFile, header = 0, delimiter=',', skipinitialspace = True)
mycolumns = ['Linear_Sequence', 'seq_len', 'nof_HBD', 'nof_HBA', 'nof_RotB', 'molPSA', 'molVolume', 'molWeight']
for i in range(2048):
    idx = str(i+1)
    mycolumns.append("ecfp" + idx)

df4.columns =mycolumns
df4= df4.drop_duplicates(subset=mycolumns,  keep='first')
#df4 = df4.drop(['molVolume', 'molWeight'], axis=1)   # Dropped only in logD code
df5=pd.merge(df3, df4, how='inner', on='Linear_Sequence')

# Bin compounds to classes 0, 1,2 based on CE property: <10: 0, 10-100: 1, .100, 2
df5['Class'] = np.where(
    df5['CE'] <= 10, 0, np.where(
    ((df5['CE'] > 10) & (df5['CE'] <=100)), 1, 2))

col_names = list(df5.columns)
n = len(col_names)
feature_cols = col_names[2:n-1]

X=df5[feature_cols].to_numpy()
y=df5['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
qt = QuantileTransformer(output_distribution='uniform', copy=False, random_state=0).fit(X_train)
X_train_transformed = qt.transform(X_train)
X_test_transformed = qt.transform(X_test)

from sklearn.utils import class_weight
classes_weights = class_weight.compute_sample_weight(class_weight='balanced', y=y_train)

model = xgb.XGBClassifier(num_class=3, sample_weight=classes_weights, use_label_encoder=False)
param_dist = {'objective':['multi:softmax'],
              'learning_rate': [0.01, 0.05],
              'max_depth': [3, 4, 5],
              'min_child_weight': [1, 2, 3, 4],
              'gamma': [0.0, 0.1, 0.2, 0.3, 0.4],
              'subsample': stats.uniform(0.3, 0.6),
              'colsample_bytree': stats.uniform(0.5, 0.4),
              'n_estimators': [500, 1000]}

random_search = RandomizedSearchCV(model, param_dist, cv=5, n_iter=5, refit=True, n_jobs=-1, verbose=3)
random_search.fit(X_train_transformed,y_train)
random_search.best_estimator_
y_pred= random_search.best_estimator_.predict(X_test_transformed)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm) # rows are predictions, cols are truth cm[0][0] = TP cm[0][1] = FP cm[1][0] = FN
print(classification_report(y_test, y_pred))


y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
y_predict_proba=random_search.best_estimator_.predict_proba(X_test_transformed)

fpr = {}
tpr = {}
thresh = {}
roc_auc = dict()

n_class = 3

for i in range(n_class):
    fpr[i], tpr[i], thresh[i] = roc_curve(y_test_binarized[:, i], y_predict_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
print(roc_auc)

import pickle
with open('CE_OHnorm1_classification_model.pkl', 'wb') as files:
    pickle.dump(random_search.best_estimator_, files)

with open('CE_OHnorm1_classification_model_QT.pkl', 'wb') as files:
    pickle.dump(qt, files)

#feature importance
X=df5[feature_cols]
y=df5['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

feature_imp = pd.DataFrame(sorted(zip(random_search.best_estimator_.feature_importances_,X_test.columns)), columns=['Importance','Feature'])
sorted_feature_imp = feature_imp.sort_values(by= "Importance", ascending = False)
sorted_feature_imp.to_csv("~/NewModels/sorted_feature_imp.csv", index= False)

#if "CE" in output.columns:
 #   bp=output.boxplot(column='CE', by='CE_classification_model_082222')
  #  bp.savefig("performance.png")

